# Campaña bancaria: estructura inicial del dataset
Carolina Machado | Data Science 1 | Preentrega 3

Continuidad: mismo Bank Marketing de UCI utilizado en la preentrega 2. Pregunta: ¿qué factores previos al contacto se asocian con la contratación de un depósito?

Fuente: Moro, S., Rita, P. y Cortez, P. (2014). Bank Marketing. DOI: 10.24432/C5K306. Licencia CC BY 4.0. Se usa bank-full.csv, no bank-additional-full.csv.

## 1. Carga reproducible
El CSV original se conserva sin modificar. El separador es punto y coma. La carpeta data incluye el diccionario original. Los resultados siguientes fueron ejecutados sobre el archivo descargado de UCI.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import gzip
pd.set_option('display.width', 110)
ruta = Path('data/bank-full.csv')
if not ruta.exists():
    # El original se conserva comprimido, sin cambiar ninguna observación.
    ruta.write_bytes(gzip.decompress(ruta.with_suffix('.csv.gz').read_bytes()))
df = pd.read_csv(ruta, sep=';')
assert df.shape == (45211, 17)
print('SHA256:', hashlib.sha256(ruta.read_bytes()).hexdigest())
print('Filas y columnas:', df.shape)
print(df.head(3).T.to_string())

SHA256: d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d
Filas y columnas: (45211, 17)
                    0           1             2
age                58          44            33
job        management  technician  entrepreneur
marital       married      single       married
education    tertiary   secondary     secondary
default            no          no            no
balance          2143          29             2
housing           yes         yes           yes
loan               no          no           yes
contact       unknown     unknown       unknown
day                 5           5             5
month             may         may           may
duration          261         151            76
campaign            1           1             1
pdays              -1          -1            -1
previous            0           0             0
poutcome      unknown     unknown       unknown
y                  no          no            no


## 2. Diagnóstico estructural
Se inspeccionan tipos y nulos técnicos. La etiqueta unknown representa información semánticamente no disponible; no equivale a un nulo de Pandas. pdays=-1 significa sin contacto previo, no un valor negativo imposible.

In [2]:
df.info()
diagnostico = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'unknown': df.eq('unknown').sum(),
})
print(diagnostico.to_string())
print('Duplicados exactos:', df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB
             tipo  nulos  unknown
age         int64      0        0
job        object      0      2

## 3. Resumen numérico
El saldo promedio anual se expresa en euros. Se conservan los saldos negativos y extremos porque pueden corresponder a situaciones reales. La duración de la llamada se describe para auditar el origen, pero se excluye de la base predictiva por conocerse después del contacto.

In [3]:
print(df.describe().T.round(2).to_string())
print('Tasa observada de contratación (%):', round(df['y'].eq('yes').mean()*100, 2))
print('Rango de saldo (EUR):', df['balance'].min(), df['balance'].max())

            count     mean      std     min    25%    50%     75%       max
age       45211.0    40.94    10.62    18.0   33.0   39.0    48.0      95.0
balance   45211.0  1362.27  3044.77 -8019.0   72.0  448.0  1428.0  102127.0
day       45211.0    15.81     8.32     1.0    8.0   16.0    21.0      31.0
duration  45211.0   258.16   257.53     0.0  103.0  180.0   319.0    4918.0
campaign  45211.0     2.76     3.10     1.0    1.0    2.0     3.0      63.0
pdays     45211.0    40.20   100.13    -1.0   -1.0   -1.0    -1.0     871.0
previous  45211.0     0.58     2.30     0.0    0.0    0.0     0.0     275.0
Tasa observada de contratación (%): 11.7
Rango de saldo (EUR): -8019 102127


## 4. Tres filtros booleanos de validación
1. Edad adulta y plausible entre 18 y 100 años.
2. Al menos un contacto en la campaña actual.
3. Recencia válida: -1 (sin contacto previo) o un número no negativo.

Son controles de dominio, no una afirmación de errores inexistentes. Se registra cuántas filas excluye cada paso; un control válido puede excluir cero.

In [4]:
pasos = []
base = df.copy()
filtros = [
    ('edad entre 18 y 100', lambda x: x['age'].between(18, 100)),
    ('campaign mayor o igual a 1', lambda x: x['campaign'].ge(1)),
    ('pdays válido', lambda x: x['pdays'].eq(-1) | x['pdays'].ge(0)),
]
for nombre, filtro in filtros:
    antes = len(base)
    base = base.loc[filtro(base)].copy()
    pasos.append((nombre, antes, len(base), antes-len(base)))
print(pd.DataFrame(pasos, columns=['control','antes','despues','excluidas']).to_string(index=False))
base = base.drop_duplicates().reset_index(drop=True)
base = base.drop(columns=['duration'])
print('Base sin duration:', base.shape)
assert 'duration' not in base

                   control  antes  despues  excluidas
       edad entre 18 y 100  45211    45211          0
campaign mayor o igual a 1  45211    45211          0
              pdays válido  45211    45211          0
Base sin duration: (45211, 16)


## 5. Transformaciones y agregaciones
Se conserva unknown como categoría explícita para evitar una imputación arbitraria. La recencia se deriva sin convertir el código -1 en días reales. Las tasas son descriptivas, no estimaciones causales.

In [5]:
base['contacto_previo_grupo'] = np.select(
    [base['pdays'].eq(-1), base['pdays'].between(0, 90)],
    ['sin contacto previo', 'contacto reciente'],
    default='contacto lejano')
base['job_normalizado'] = base['job'].map(lambda x: x.strip().lower())
base['suscribe'] = base['y'].eq('yes').astype(int)
for columna in ['job_normalizado', 'education', 'contacto_previo_grupo']:
    resumen = base.groupby(columna)['suscribe'].agg(n='size', tasa='mean')
    resumen['tasa_pct'] = (resumen.pop('tasa')*100).round(2)
    print('\nAgrupación:', columna)
    print(resumen.to_string())
base.to_csv('data/bank_saneado.csv', index=False)
print('CSV saneado guardado:', base.shape)


Agrupación: job_normalizado
                    n  tasa_pct
job_normalizado                
admin.           5171     12.20
blue-collar      9732      7.27
entrepreneur     1487      8.27
housemaid        1240      8.79
management       9458     13.76
retired          2264     22.79
self-employed    1579     11.84
services         4154      8.88
student           938     28.68
technician       7597     11.06
unemployed       1303     15.50
unknown           288     11.81

Agrupación: education
               n  tasa_pct
education                 
primary     6851      8.63
secondary  23202     10.56
tertiary   13301     15.01
unknown     1857     13.57

Agrupación: contacto_previo_grupo
                           n  tasa_pct
contacto_previo_grupo                 
contacto lejano         7539     21.94
contacto reciente        718     34.96
sin contacto previo    36954      9.16
CSV saneado guardado: (45211, 19)


## 6. Conclusión y límites
No se detectaron nulos técnicos ni duplicados exactos. Sí hay categorías unknown y el código pdays=-1, que se documentan y conservan. Los controles no justifican eliminar saldos extremos. Variables clave: balance, campaign, previous, poutcome, contact y sus relaciones con y. age y job se estudian descriptivamente y requieren evaluar sesgos antes de cualquier uso operativo.

Se excluye duration por fuga de información. No se reconstruye una fecha completa con day y month porque falta el año por registro. En un modelo posterior habrá que precisar el momento de predicción y qué variables de campaña están disponibles en ese instante. Este análisis corresponde a un banco y un período históricos; no representa a todos los clientes ni prueba causalidad. No se entrenó ni desplegó un modelo.